# Parser agent tests

Run cells top to bottom.

**Setup:** copy `.env.example` to `.env` and add your Anthropic API key:

```bash
cp .env.example .env
```

Install dev dependencies if needed: `pip install -e ".[dev]"`

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "orharness").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Copy .env.example to .env and add your key."
    )

print("API key loaded from .env")

In [ ]:
from orharness.models import ORHarnessConfig
from orharness.agents.parser import parse_problem

config = ORHarnessConfig()
print(config)

## Smoke test (optional)

Quick check that the API key and model name work before calling the parser.

In [ ]:
from litellm import completion

response = completion(
    model=config.model,
    messages=[{"role": "user", "content": "Reply with exactly: ok"}],
    max_tokens=16,
)
print(response.choices[0].message.content)

## Test: valid scheduling problem

Should return a `ParsedProblem` with `is_or_problem=True`, `problem_type=scheduling`,
and `objective_kind=feasibility` for this input.

In [ ]:
result = parse_problem(
    "I have 6 nurses, 3 shifts per day, 7 days a week. "
    "No nurse works more than 5 shifts per week. "
    "Night shifts need at least 2 nurses.",
    config,
)
print(result)

## Test: not an optimization problem

General questions should raise `ClassificationError`.

In [ ]:
try:
    parse_problem("What's the best way to learn Python?", config)
except Exception as e:
    print(type(e).__name__, ":", e)

## Test: too vague

Ambiguous inputs should also raise `ClassificationError`.

In [ ]:
try:
    parse_problem("I need to organize my team somehow", config)
except Exception as e:
    print(type(e).__name__, ":", e)